# Geração de Dados Sintéticos — `raw.pagamentos`

**Objetivo:** Popular a tabela `raw.pagamentos` com **21.500 registros**.

**Regras de integridade:**
- `id_transacao_raw` referencia transações da tabela fato `raw.transacoes_financeiras`.
- São consideradas preferencialmente transações do tipo **DESPESA** com status **PAGO** ou **ATRASADO**.
  - DESPESA PAGO: 19.464 registros → todos incluídos.
  - DESPESA ATRASADO: 2.937 disponíveis → 2.036 amostrados para completar 21.500.
- `id_transacao_raw` é o sufixo numérico do TR-XXXXX (e.g., TR-00140 → 140).
- `data_pagamento` é a `data_transacao` da fato + 0–5 dias úteis.
- `valor_pago` é o `valor_liquido` da fato (com pequena variação para ATRASADO).

**Reprodutibilidade:** `seed = 42`

In [ ]:
# ============================================================
# 1. IMPORTS E CONFIGURAÇÕES
# ============================================================
import pandas as pd
import numpy as np
import hashlib
import uuid
import os
import glob
import random
from datetime import datetime, timedelta

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

QTD_PAGAMENTOS = 21_500
SOURCE_SYSTEM  = 'ERP_CORPORATIVO'
SOURCE_ENTITY  = 'pagamentos'
INGESTION_ID   = str(uuid.uuid4())
INGESTION_TS   = datetime(2026, 1, 10, 9, 30, 0).strftime('%Y-%m-%dT%H:%M:%S.000Z')

# print(f'ingestion_id : {INGESTION_ID}')
# print(f'ingestion_ts : {INGESTION_TS}')

ingestion_id : 10bc98d1-da48-4750-9c17-bbdb1c75d6e2
ingestion_ts : 2026-01-10T09:30:00.000Z


In [ ]:
# ============================================================
# 2. LEITURA DA TABELA FATO
# ============================================================
workspace = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
fato_dir  = os.path.join(workspace, 'data', 'raw', 'transacoes_financeiras')
fato_csvs = sorted(glob.glob(os.path.join(fato_dir, '*.csv')))

dfs = [pd.read_csv(f, usecols=[
    'id_transacao_raw', 'data_transacao', 'valor_liquido',
    'forma_pagamento', 'tipo_transacao', 'status_pagamento'
]) for f in fato_csvs]
df_fato = pd.concat(dfs, ignore_index=True)

# Extrair sufixo numérico: TR-00140 -> 140
df_fato['id_transacao_num'] = df_fato['id_transacao_raw'].str.replace('TR-', '').astype(int)
df_fato['data_transacao']   = pd.to_datetime(df_fato['data_transacao'])
df_fato['valor_liquido']    = df_fato['valor_liquido'].astype(float)

print(f'Tabela fato carregada: {len(df_fato):,} registros')
print(df_fato.groupby(['tipo_transacao', 'status_pagamento']).size().unstack(fill_value=0))

Tabela fato carregada: 39,792 registros
status_pagamento  ATRASADO  CANCELADO   PAGO
tipo_transacao                              
DESPESA               2937       1409  19464
RECEITA               2629       1011  12342


In [3]:
# ============================================================
# 3. SELEÇÃO DAS TRANSAÇÕES DE REFERÊNCIA
# ============================================================

# PAGO DESPESA — todos incluídos
df_pago = df_fato[
    (df_fato['tipo_transacao'] == 'DESPESA') &
    (df_fato['status_pagamento'] == 'PAGO')
].copy()

# ATRASADO DESPESA — amostrar para completar 21.500
qtd_atrasado_necessario = QTD_PAGAMENTOS - len(df_pago)
df_atrasado = df_fato[
    (df_fato['tipo_transacao'] == 'DESPESA') &
    (df_fato['status_pagamento'] == 'ATRASADO')
].sample(n=min(qtd_atrasado_necessario, len(df_fato[
    (df_fato['tipo_transacao'] == 'DESPESA') &
    (df_fato['status_pagamento'] == 'ATRASADO')
])), random_state=SEED).copy()

df_base = pd.concat([df_pago, df_atrasado], ignore_index=True)
# Se ainda faltar, completar com CANCELADO DESPESA
if len(df_base) < QTD_PAGAMENTOS:
    faltam = QTD_PAGAMENTOS - len(df_base)
    df_canc = df_fato[
        (df_fato['tipo_transacao'] == 'DESPESA') &
        (df_fato['status_pagamento'] == 'CANCELADO')
    ].sample(n=min(faltam, len(df_fato[
        (df_fato['tipo_transacao'] == 'DESPESA') &
        (df_fato['status_pagamento'] == 'CANCELADO')
    ])), random_state=SEED).copy()
    df_base = pd.concat([df_base, df_canc], ignore_index=True)

df_base = df_base.head(QTD_PAGAMENTOS).reset_index(drop=True)

print(f'Transações selecionadas: {len(df_base):,}')
print(f'  PAGO:      {len(df_pago):,}')
print(f'  ATRASADO:  {len(df_atrasado):,}')

Transações selecionadas: 21,500
  PAGO:      19,464
  ATRASADO:  2,036


In [4]:
# ============================================================
# 4. GERAÇÃO DOS REGISTROS DE PAGAMENTO
# ============================================================

METODOS_PAGAMENTO = ['PIX', 'TED', 'BOLETO', 'CARTAO', 'TRANSFERENCIA', 'CHEQUE']
PESOS_METODO      = [0.40, 0.20, 0.20, 0.10, 0.07, 0.03]

def gerar_comprovante(id_pag, id_trans, data_str):
    conteudo = f'{id_pag}-{id_trans}-{data_str}'
    return 'COMP-' + hashlib.md5(conteudo.encode()).hexdigest()[:12].upper()

def gerar_hash(row_dict):
    campos = ['id_pagamento_raw', 'id_transacao_raw', 'data_pagamento', 'valor_pago', 'metodo_pagamento']
    conteudo = '|'.join(str(row_dict.get(c, '')) for c in campos)
    return hashlib.sha256(conteudo.encode('utf-8')).hexdigest()

rng_metodo = np.random.RandomState(SEED)
rng_dias   = np.random.RandomState(SEED + 1)
rng_valor  = np.random.RandomState(SEED + 2)

registros = []
for seq, row in df_base.iterrows():
    id_pag       = seq + 1
    id_trans_int = int(row['id_transacao_num'])
    status       = row['status_pagamento']

    # Data de pagamento: data_transacao + 0 a 5 dias (ATRASADO pode ter +5 a +30)
    if status == 'ATRASADO':
        delta_dias = rng_dias.randint(5, 31)
    else:
        delta_dias = rng_dias.randint(0, 6)
    data_pag = (row['data_transacao'] + timedelta(days=int(delta_dias))).strftime('%Y-%m-%d')

    # Valor pago: mesmo que valor_liquido (variação ±2% para ATRASADO com juros)
    if status == 'ATRASADO':
        fator = 1.0 + rng_valor.uniform(0.005, 0.02)
        valor = round(float(row['valor_liquido']) * fator, 2)
    else:
        valor = round(float(row['valor_liquido']), 2)

    # Método de pagamento: usar o mesmo da fato quando disponível, ou sortear
    metodo_fato = str(row.get('forma_pagamento', '')).upper()
    if metodo_fato in METODOS_PAGAMENTO:
        metodo = metodo_fato
    else:
        metodo = rng_metodo.choice(METODOS_PAGAMENTO, p=PESOS_METODO)

    comprovante = gerar_comprovante(id_pag, id_trans_int, data_pag)

    reg = {
        'id_pagamento_raw':  id_pag,
        'id_transacao_raw':  id_trans_int,
        'data_pagamento':    data_pag,
        'valor_pago':        valor,
        'metodo_pagamento':  metodo,
        'comprovante':       comprovante,
    }
    reg['raw_row_hash'] = gerar_hash(reg)
    registros.append(reg)

print(f'Registros de pagamento gerados: {len(registros):,}')

Registros de pagamento gerados: 21,500


In [5]:
# ============================================================
# 5. CONSOLIDAÇÃO E METADADOS
# ============================================================

for seq, row in enumerate(registros, start=1):
    row['ingestion_id']  = INGESTION_ID
    row['ingestion_ts']  = INGESTION_TS
    row['source_system'] = SOURCE_SYSTEM
    row['source_entity'] = SOURCE_ENTITY
    row['row_seq']       = seq

COLUNAS = [
    'id_pagamento_raw', 'id_transacao_raw', 'data_pagamento', 'valor_pago',
    'metodo_pagamento', 'comprovante', 'ingestion_id', 'ingestion_ts',
    'source_system', 'source_entity', 'row_seq', 'raw_row_hash',
]
df_pag = pd.DataFrame(registros, columns=COLUNAS)

print(f'Shape final: {df_pag.shape}')
df_pag.head()

Shape final: (21500, 12)


,id_pagamento_raw,id_transacao_raw,data_pagamento,valor_pago,metodo_pagamento,comprovante,ingestion_id,ingestion_ts,source_system,source_entity,row_seq,raw_row_hash
0,1,144,2024-01-07,3089.73,PIX,COMP-4A64F6186A82,10bc98d1-da48-4750-9c17-bbdb1c75d6e2,2026-01-10T09:30:00.000Z,ERP_CORPORATIVO,pagamentos,1,7d551a1c2c3b6761145a2b9a63e591b9286d1ea75cdaf8...
1,2,146,2024-01-23,2527.31,TED,COMP-3868E3E3E080,10bc98d1-da48-4750-9c17-bbdb1c75d6e2,2026-01-10T09:30:00.000Z,ERP_CORPORATIVO,pagamentos,2,56b16e94015be4e2af7c35b6005e42688ea351da7586f4...
2,3,147,2024-01-09,2459.29,CARTAO,COMP-ED11FCF173DD,10bc98d1-da48-4750-9c17-bbdb1c75d6e2,2026-01-10T09:30:00.000Z,ERP_CORPORATIVO,pagamentos,3,2ea94f3f90d30c80cbf7d5d569eaec9ee7f153a156813a...
3,4,149,2024-01-31,2987.76,BOLETO,COMP-64C9176A96BA,10bc98d1-da48-4750-9c17-bbdb1c75d6e2,2026-01-10T09:30:00.000Z,ERP_CORPORATIVO,pagamentos,4,00c507280bbbc54364a05a73f5d1b1ce3a34482e0043e8...
4,5,150,2024-01-20,3086.49,TRANSFERENCIA,COMP-99D03E27B0D8,10bc98d1-da48-4750-9c17-bbdb1c75d6e2,2026-01-10T09:30:00.000Z,ERP_CORPORATIVO,pagamentos,5,99cfbf636b4bbf087a9ab389573c2c974adef508ea6807...


In [ ]:
# ============================================================
# 6. VALIDAÇÕES
# ============================================================

assert len(df_pag) == QTD_PAGAMENTOS, f'Esperado {QTD_PAGAMENTOS:,}, gerado {len(df_pag):,}'
assert df_pag['id_pagamento_raw'].nunique() == QTD_PAGAMENTOS, 'IDs duplicados!'
assert df_pag['comprovante'].nunique() == QTD_PAGAMENTOS, 'Comprovantes duplicados!'
assert df_pag['valor_pago'].min() > 0, 'Valores negativos ou zero!'
assert not df_pag['data_pagamento'].isna().any(), 'Datas nulas!'

# print(f'{len(df_pag):,} registros gerados.')
# print('IDs únicos, comprovantes únicos, valores positivos.')
# print()
# print('Distribuição por metodo_pagamento:')
# print(df_pag['metodo_pagamento'].value_counts().to_string())
# print()
# print('Estatísticas de valor_pago:')
# print(df_pag['valor_pago'].describe().round(2).to_string())

✔ 21,500 registros gerados.
✔ IDs únicos, comprovantes únicos, valores positivos.

Distribuição por metodo_pagamento:
metodo_pagamento
PIX              6004
BOLETO           5283
TED              4295
TRANSFERENCIA    3795
CARTAO           2123

Estatísticas de valor_pago:
count    21500.00
mean      3395.35
std       2658.62
min        155.83
25%       1829.82
50%       2617.36
75%       4046.38
max      41419.87


In [ ]:
# ============================================================
# 7. EXPORTAÇÃO PARA CSV
# ============================================================

output_dir  = os.path.join(workspace, 'data', 'raw', 'pagamentos')
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, 'pagamentos.csv')
df_pag.to_csv(output_path, index=False, encoding='utf-8')

# print(f'Arquivo exportado: {output_path}')
# print(f'Total de registros: {len(df_pag):,}')

Arquivo exportado: c:\Users\Adam\Documents\Repositorio\TCC\SCAP\data\raw\pagamentos\pagamentos.csv
Total de registros: 21,500
